# Bash 5.2 → 5.3 migration — hands-on checklist

Runnable companion to `bash/docs/bash-5.3-migration-guide.md`. Run the code cells in order on the old runtime, then re-run on the new runtime and diff the outputs before swapping a fleet.

**last_verified:** 2026-09-29  
**tool_version:** Bash 5.3 patchlevel 20 (pkgsrc bumped `BASH_PATCHLEVEL` from 15 to 20 on 2026-09-15)  
**sources:** https://mail-index.netbsd.org/pkgsrc-changes/2026/09/15/msg354675.html, https://www.golinuxcloud.com/bash-best-practices/

Version note: the companion prose guide records `tool_version: 5.3.15` in its front-matter. That is the pre-bump level of the same 15→20 window recorded by the pkgsrc notice; this checklist targets patchlevel 20, the post-bump level. Both sit on the Bash 5.3 line, so there is no cross-document contradiction — only a patchlevel delta inside one documented window.

## Purpose

Give a 5.2 → 5.3 runtime upgrade a repeatable, CI-gatable verification pass: confirm which patchlevel is under test, re-establish the production scripting baseline, and record observable behavior in the areas the migration brief names (pathname-expansion ordering, new in-shell substitution forms, `test`/`[` usage) by running minimal repros on both runtimes and comparing output — instead of discovering a difference from a post-upgrade alert.

## When to use

| Situation | Action |
|---|---|
| Auditing scripts for a 5.2 → 5.3 upgrade | Run every code cell on 5.2, save output, re-run on 5.3, diff. |
| CI on a mixed-version fleet | Gate 5.3-only syntax behind the `BASH_VERSINFO` guard in cell 0 so a 5.2 runner skips instead of failing mid-script. |
| Adopting the 5.3 patch fixes | Check cells 1–2 to confirm the patchlevel window before signing off. |

## Prerequisites

- A Bash 5.3 interpreter for the post-upgrade pass (`bash --version`); the pre-upgrade pass runs on the production baseline `bash 5.2.26-6.el10` on RHEL 10.2.
- Lint gate: `bash -n` plus ShellCheck `0.10.0` (the production-baseline version) with a representative execution test — ShellCheck alone is not a test suite.
- Standard utilities from the baseline: `coreutils 9.5-8.el10_2`, plus `mktemp`, `flock`, `timeout`, and `grep`.
- A scratch directory writable by the current user (cells allocate it with `mktemp`, never with predictable `/tmp` names).

In [ ]:
# Cell 0 — version guard. Run first; 5.3-only probes below self-skip on older runtimes.
echo "bash: $(bash --version | head -1)"
echo "BASH_VERSINFO: ${BASH_VERSINFO[0]}.${BASH_VERSINFO[1]} patchlevel ${BASH_PATCHLEVEL}"
if command -v shellcheck >/dev/null 2>&1; then shellcheck --version | head -2; else echo "shellcheck: not installed"; fi
if [ "${BASH_VERSINFO[0]}" -eq 5 ] && [ "${BASH_VERSINFO[1]}" -ge 3 ]; then echo "RUNTIME=5.3+: run all cells"; else echo "RUNTIME<5.3: 5.3-only probes self-skip"; fi

## Steps

### 1. Confirm the 5.3 patchlevel window

The pkgsrc notice of 2026-09-15 bumped `BASH_PATCHLEVEL` from 15 to 20, adding patch files `bash53-016` through `bash53-020`. Their contents:

| Patch | Content |
|---|---|
| 16 | macOS dynamic pipe sizing |
| 17 | readline redisplay buffer offset with multibyte prompts |
| 18 | readline crash on prompts exceeding 256 wrapped lines |
| 19 | `isalpha` handling for bytes 128–255 |
| 20 | readline `SIGWINCH` terminal-size recompute |

If the fleet carries anything below patchlevel 15, the upgrade spans a wider delta than this checklist covers — note it before proceeding.

In [ ]:
# Cell 1 — patchlevel window check.
pl="${BASH_PATCHLEVEL:-0}"
echo "patchlevel: $pl"
if [ "$pl" -ge 15 ] && [ "$pl" -le 20 ]; then echo "inside the 15-to-20 window from the 2026-09-15 pkgsrc bump"; else echo "outside the 15-to-20 window — record the delta before signing off"; fi

### 2. Re-establish the production scripting baseline

The baseline this checklist was tested against is `bash 5.2.26-6.el10`, ShellCheck `0.10.0`, `coreutils 9.5-8.el10_2` on RHEL 10.2. Before auditing any 5.3 delta, confirm the habits that keep exit statuses honest on both runtimes:

- Build argument lists as arrays and expand with `"${args[@]}"`; quote expansions deliberately (`"$var"`, `"${array[@]}"`, `"$@"`).
- Validate binaries and required variables up front with clear stderr messages.
- Keep a stdout/stderr/exit contract (stdout machine-clean, diagnostics to stderr); handle expected non-zero statuses explicitly (`grep` exit 1 means no match, greater than 1 means a real error).
- Use `set -euo pipefail` as a documented baseline, not a magic switch (`set -e` does not exit inside `if` tests or `||` chains; `set -u` needs `${VAR:-}` defaults); register `trap` cleanup immediately after allocating temp files.
- Design scheduled steps idempotent (safe to run twice); guard them with non-blocking `flock` (`flock -n` contention exits 1 by default; use `flock -E` for a documented conflict code); bound network calls with `timeout` (GNU `timeout` returns 124 on expiry).

In [ ]:
# Cell 2 — validate binaries and required variables up front.
for bin in bash shellcheck mktemp flock timeout grep; do
  if command -v "$bin" >/dev/null 2>&1; then echo "ok: $bin -> $(command -v "$bin")";
  else echo "MISSING: $bin" >&2; fi
done

In [ ]:
# Cell 3 — arrays and deliberate quoting.
args=(--strict --jobs 4)
printf 'arg count: %s\n' "$#"
printf '[%s]\n' "${args[@]}"
src="a b"
printf 'quoted: [%s]\n' "$src"

In [ ]:
# Cell 4 — grep exit-status contract: 1 is no-match, greater than 1 is a real error.
printf 'a\nb\n' | grep -q z; echo "no-match exit: $?"
printf 'a\nb\n' | grep -q a; echo "match exit: $?"
grep -q z /nonexistent-file-xyz 2>/dev/null; echo "real-error exit: $?"

In [ ]:
# Cell 5 — trap cleanup registered right after mktemp allocation.
tmpdir=$(mktemp -d)
trap 'rm -rf "$tmpdir"' EXIT
touch "$tmpdir/fixture"
ls "$tmpdir"
trap - EXIT
rm -rf "$tmpdir"
echo "scratch cleaned"

### 3. Audit the brief-named areas by observation, not by assumption

The migration brief names three areas to check: pathname-expansion ordering (glob sorting), the new in-shell substitution forms, and `test`/`[` usage. The cells below do not assert what the 5.3 behavior is — they produce output to diff across runtimes:

- **Glob ordering:** run cell 6 on 5.2, save the listing, re-run on 5.3, diff. Any reorder is a migration finding for scripts that parse file listings.
- **Substitution exit status:** cell 7 records how the portable `$(...)` form propagates a failing exit status on the current runtime, then — only on a 5.3 runtime — invites re-running the same probe through the new in-shell form and diffing `$?` handling.
- **`test`/`[` usage:** cell 8 smokes simple chains under both `[ ]` and `[[ ]]`, then directs the full existing suite (not ShellCheck alone) to be re-run under 5.3, since only execution catches behavioral drift.

In [ ]:
# Cell 6 — glob-ordering fixture. Save output on 5.2, re-run on 5.3, diff.
g=$(mktemp -d)
touch "$g/1-access.log" "$g/10-access.log" "$g/2-access.log"
(cd "$g" && printf '%s\n' *.log)
rm -rf "$g"

In [ ]:
# Cell 7 — substitution exit-status probe (portable part) + guarded 5.3 prompt.
out=$(printf 'hi'; exit 3); echo "portable-form exit: $? (output was: $out)"
if [ "${BASH_VERSINFO[0]}" -eq 5 ] && [ "${BASH_VERSINFO[1]}" -ge 3 ]; then echo "5.3 runtime: re-run this probe through the new in-shell substitution form here and diff \$? handling"; else echo "SKIP: in-shell substitution probe needs Bash 5.3 (current: $BASH_VERSION)"; fi

In [ ]:
# Cell 8 — test-builtin smoke on simple chains; complex expressions go through the suite.
a=5; b=7
[ "$a" -gt 0 ] && [ "$b" -lt 10 ] && echo "simple [ ] chain: ok"
[[ $a -gt 0 && $b -lt 10 ]] && echo "[[ ]] chain: ok"
echo "complex parenthesized expressions: re-run your own suite under both runtimes and diff"

## Verify

Gate every change on `bash -n`, ShellCheck, and a representative execution test — ShellCheck alone is not a test suite:

1. Confirm the interpreter: `bash --version` prints the expected 5.3 patchlevel (cell 0).
2. Run cells 1–8 on the old runtime, save all outputs; re-run on the new runtime; diff.
3. Lint with the baseline gate: `bash -n` on each cell's code plus `shellcheck -s bash` at ShellCheck `0.10.0`; review any finding, documenting suppressions rather than chasing zero warnings.
4. In CI, keep 5.3-only syntax behind the cell-0 version guard so a 5.2 runner skips instead of failing mid-script.

In [ ]:
# Cell 9 — the lint gate itself, against the baseline ShellCheck 0.10.0.
bash -n <<< 'echo ok' && echo "bash -n: clean"
printf 'x=1\necho "$x"\n' | shellcheck -s bash - && echo "shellcheck: clean"

## Rollback

Keep the old runtime available until the diff in Verify is empty:

1. Pin the previous interpreter path (e.g. the `bash 5.2.26-6.el10` binary) alongside the new one during the migration window.
2. If any cell-6/7/8 diff is unexplained, point job shebangs and CI images back at the old binary and re-run the checklist — every step is idempotent and safe to run twice.
3. Re-promote to 5.3 only after `bash -n`, ShellCheck `0.10.0`, and the execution diff are all green.

## Common errors

- **Running the 5.3 probes on a 5.2 kernel:** the guarded prompts print `SKIP` instead — that is the guard working, not a failure. Read the skip line before re-running.
- **Treating a clean ShellCheck run as a passing migration:** the baseline guidance is explicit that ShellCheck alone is not a test suite. A green lint with a red execution diff still blocks the rollout.
- **Parsing `ls`-style output downstream of cell 6:** if the 5.2/5.3 listings differ, fix the consumer to sort explicitly instead of relying on expansion order.
- **Ignoring `$?` after substitution:** the portable probe in cell 7 shows the inner failure surfacing as the assignment status — code that never inspected `$?` after `$(...)` may branch differently once the same call site uses an in-shell form.

## References

- Prose reference for the same upgrade: `bash/docs/bash-5.3-migration-guide.md` (records `tool_version: 5.3.15`, the pre-bump level of the 15→20 window).
- Bash 5.3 patchlevel 20 and patch 16–20 contents: https://mail-index.netbsd.org/pkgsrc-changes/2026/09/15/msg354675.html
- Production scripting baseline and checklist patterns (`bash 5.2.26-6.el10`, ShellCheck `0.10.0`, `coreutils 9.5-8.el10_2` on RHEL 10.2): https://www.golinuxcloud.com/bash-best-practices/